# 🗄️ Cosmos DB — case state, measured

**Birlasoft FORGE FDE Academy · Sprint 1 Day 18 · DecisionStream AI**

---

Three query patterns want three different partition keys. You get one, and
**it is immutable** — fixed when the container is created, and changing it
means migrating the data.

> Almost everything in this programme can be changed later. This cannot.
> It is the one decision today that you either get right or live with.

### Why this session is different from the others

Every other notebook in this sprint has modelled something. This one does not
need to.

**Cosmos returns the RU charge on every single response.** `x-ms-request-charge`
is in the headers of every read, query and write. So every claim in the deck
becomes something you measure against your own account:

| Finding | How it is established here |
|---|---|
| No partition key wins all three patterns | **Measured** — three real containers, real RU |
| A cross-partition query charges per document scanned | **Measured** |
| Partition skew, and where the hot partition comes from | **Measured** |
| Splitting hot from cold cuts write RU | **Measured** |
| The change feed loses intermediate states | **Observed** — against your account |

Nothing here is a simulation. If a number surprises you, it is your account
telling you something.

---

### Before you run anything

**Your key is a credential.** Masked prompt, never printed. A shared Colab
notebook carries its outputs.

**This creates containers in your account.** Three of them, plus two more for
the document-shape experiment. The final cell deletes everything it made —
**run it.** On a provisioned account each container may carry a 400 RU/s
minimum, so leaving five behind is a real monthly charge.

**Cost.** The defaults load **400 cases** rather than 4,000, which is a few
thousand RU and pennies on serverless. Raise `N_CASES` once it works.

## 1 · Install and import

In [ ]:
%pip install -q azure-cosmos

import os, re, json, time, uuid, random, getpass, warnings, statistics
from collections import defaultdict, Counter
from datetime import datetime, timezone

from azure.cosmos import CosmosClient, PartitionKey, exceptions

warnings.filterwarnings("ignore")
print("imports ready")

## 2 · Your Cosmos credentials

Portal → your Cosmos DB account → **Keys** → URI and Primary Key.

This must be a **Core (SQL) API** account. If you created a MongoDB or
Cassandra API account, the SDK below will not talk to it.

In [ ]:
COSMOS_URL = os.environ.get("COSMOS_URL", "").strip()
COSMOS_KEY = os.environ.get("COSMOS_KEY", "").strip()

if not COSMOS_URL:
    COSMOS_URL = input("Cosmos DB URI (https://<account>.documents.azure.com:443/): ").strip()
if COSMOS_URL and not COSMOS_KEY:
    COSMOS_KEY = getpass.getpass("Primary key (hidden): ").strip()

LIVE = bool(COSMOS_URL and COSMOS_KEY)

DB_NAME   = "forge_day18"
N_CASES   = 400          # raise once it works. 4000 is the deck's figure.
PREFIX    = "ds"         # container name prefix, so cleanup is unambiguous

client = db = None
if LIVE:
    client = CosmosClient(COSMOS_URL, credential=COSMOS_KEY)
    db = client.create_database_if_not_exists(id=DB_NAME)
    print(f"mode:     LIVE")
    print(f"account:  {COSMOS_URL.split('//')[-1].split('.')[0]}")
    print(f"database: {DB_NAME}")
    print(f"cases:    {N_CASES}")
    print("\nContainers created by this notebook are prefixed "
          f"'{PREFIX}_' and are deleted by the final cell. Run it.")
else:
    print("mode: SIMULATED — no account touched, RU figures are modelled.")
    print("Every finding still appears; the conviction is weaker.")

## 3 · The RU meter

The whole session depends on reading one header. This helper captures it for
reads, queries and writes.

`x-ms-request-charge` is the number. Not milliseconds — **RU is the currency,
and the bill is RU.**

In [ ]:
def q(container, query, params=None, cross=True):
    """
    Query wrapper that works across azure-cosmos versions.

    `enable_cross_partition_query` was required in older SDKs and removed in
    newer ones, where cross-partition is the default. Passing it to a recent
    SDK raises. This tries with, falls back to without.

    That is not defensive programming for its own sake — it is the same
    method-over-frozen-values point the programme keeps making. The parameter
    name is a vendor detail; the behaviour is what you are teaching.
    """
    kw = {"query": query}
    if params:
        kw["parameters"] = params
    try:
        return list(container.query_items(**kw, enable_cross_partition_query=cross))
    except TypeError:
        return list(container.query_items(**kw))


def read_changefeed(container, from_beginning=True):
    """Change feed reader, tolerant of SDK signature changes."""
    for attempt in (
        lambda: container.query_items_change_feed(is_start_from_beginning=from_beginning),
        lambda: container.query_items_change_feed(start_time="Beginning" if from_beginning else "Now"),
        lambda: container.query_items_change_feed(),
    ):
        try:
            return list(attempt())
        except (TypeError, ValueError):
            continue
    return []


def charge(container):
    """RU consumed by the most recent operation on this container."""
    try:
        h = container.client_connection.last_response_headers
        return float(h.get("x-ms-request-charge", 0.0))
    except Exception:
        return 0.0

def timed(fn, *a, **kw):
    """Run an operation, return (result, ru, ms)."""
    t0 = time.perf_counter()
    out = fn(*a, **kw)
    if hasattr(out, "__iter__") and not isinstance(out, (dict, list, str)):
        out = list(out)                     # exhaust the iterator, or RU is 0
    return out, None, (time.perf_counter() - t0) * 1000

STATUSES = ["pending", "extracting", "generating", "complete", "needs_human"]
# Deliberately lopsided. Most cases finish, which is why `status` becomes a
# hot partition key later.
STATUS_WEIGHTS = [0.06, 0.04, 0.03, 0.74, 0.13]

def make_case(i, rng, fat=True):
    cid = f"CLM-2026-{4000 + i}"
    aid = f"APP-{rng.randint(1, max(1, N_CASES // 3)):05d}"
    st  = rng.choices(STATUSES, STATUS_WEIGHTS)[0]
    doc = {"id": cid, "case_id": cid, "applicant_id": aid, "status": st,
           "updated_at": datetime.now(timezone.utc).isoformat()}
    if fat:
        doc["extracted_fields"] = {f"field_{k}": f"value_{rng.randint(1000,9999)}"
                                   for k in range(18)}
        doc["ai_response"] = {
            "narrative": "The claim relates to a front nearside collision. " * 24,
            "citations": [{"clause": f"{k}.{k}", "page": k, "quote": "x" * 90}
                          for k in range(8)],
            "risk_indicators": ["estimate_exceeds_market_value",
                                "supplementary_above_threshold"]}
        doc["model_version"] = "gpt-4o-mini@2026-07-01"
    else:
        doc["result_ref"] = f"results/{cid}"
    return doc

rng = random.Random(7)
CASES = [make_case(i, rng) for i in range(N_CASES)]
print(f"{len(CASES)} synthetic cases prepared")
print(f"document size: ~{len(json.dumps(CASES[0]))/1024:.1f} KB")
print(f"status mix:    {dict(Counter(c['status'] for c in CASES))}")

## 4 · Three containers, three partition keys

The same 400 documents, loaded three times, partitioned three different ways.

This is the only slow cell — it writes 1,200 documents. Watch the write RU: it
is the same data every time, so any difference is the partition key, not the
payload.

In [ ]:
CANDIDATES = ["case_id", "applicant_id", "status"]
containers, load_stats = {}, {}

if LIVE:
    for pk in CANDIDATES:
        name = f"{PREFIX}_by_{pk}"
        try:
            db.delete_container(name)          # clean slate if re-run
        except exceptions.CosmosResourceNotFoundError:
            pass
        c = db.create_container(id=name, partition_key=PartitionKey(path=f"/{pk}"))
        containers[pk] = c

        ru_total, t0 = 0.0, time.perf_counter()
        for doc in CASES:
            c.upsert_item(doc)
            ru_total += charge(c)
        secs = time.perf_counter() - t0
        load_stats[pk] = {"ru": ru_total, "secs": secs,
                          "ru_per_write": ru_total / len(CASES)}
        print(f"  {name:<22} {len(CASES)} docs   {ru_total:>9,.0f} RU   "
              f"{ru_total/len(CASES):>5.1f} RU/write   {secs:>5.1f}s")
else:
    for pk in CANDIDATES:
        load_stats[pk] = {"ru": N_CASES * 14.8, "secs": 0.0, "ru_per_write": 14.8}
        print(f"  {PREFIX}_by_{pk:<14} {N_CASES} docs   "
              f"{N_CASES*14.8:>9,.0f} RU   14.8 RU/write   (simulated)")

print(f"""
Write RU is essentially identical across the three, and that is the point:
write cost follows DOCUMENT SIZE, not the partition key.

The partition key decides what your READS cost. That is next.
""")

## 5 · The three query patterns — measured

Your brief lists three:

| Pattern | Volume | What it is |
|---|---|---|
| `get by case_id` | 40,000 / month | the hot path |
| `list by applicant_id` | 3,000 / month | a handler checking history |
| `filter by status` | 6,000 / month | the operations queue |

Run each against each container and read the real RU charge.

**Watch the point read.** When the partition key is in the query, Cosmos knows
where to go. When it is not, it visits every partition and charges you per
document examined.

In [ ]:
SAMPLES = 12

def run_patterns(pk, container, index):
    out = {}
    r = random.Random(11)

    # --- get by case_id ---
    rus = []
    for _ in range(SAMPLES):
        row = r.choice(index)
        if pk == "case_id":
            container.read_item(item=row["case_id"], partition_key=row["case_id"])
        else:
            q(container, "SELECT * FROM c WHERE c.case_id=@v",
              [{"name": "@v", "value": row["case_id"]}])
        rus.append(charge(container))
    out["get_by_case_id"] = statistics.median(rus)

    # --- list by applicant_id ---
    rus = []
    for _ in range(SAMPLES):
        row = r.choice(index)
        q(container, "SELECT * FROM c WHERE c.applicant_id=@v",
          [{"name": "@v", "value": row["applicant_id"]}], cross=(pk != "applicant_id"))
        rus.append(charge(container))
    out["list_by_applicant"] = statistics.median(rus)

    # --- filter by status ---
    rus = []
    for _ in range(SAMPLES):
        st = r.choices(STATUSES, STATUS_WEIGHTS)[0]
        q(container, "SELECT * FROM c WHERE c.status=@v",
          [{"name": "@v", "value": st}], cross=(pk != "status"))
        rus.append(charge(container))
    out["filter_by_status"] = statistics.median(rus)
    return out

SIM = {  # scaled to N_CASES, from the offline lab
  "case_id":      {"get_by_case_id": 1.5,   "list_by_applicant": N_CASES*0.46,
                   "filter_by_status": N_CASES*0.46},
  "applicant_id": {"get_by_case_id": N_CASES*0.46, "list_by_applicant": 4.3,
                   "filter_by_status": N_CASES*0.46},
  "status":       {"get_by_case_id": N_CASES*0.46, "list_by_applicant": N_CASES*0.46,
                   "filter_by_status": N_CASES*0.26},
}

PATTERNS = {}
print(f"{'partition key':<16}{'get by case_id':>17}{'list by applicant':>20}"
      f"{'filter by status':>19}")
print("-" * 72)
for pk in CANDIDATES:
    PATTERNS[pk] = run_patterns(pk, containers[pk], CASES) if LIVE else SIM[pk]
    p = PATTERNS[pk]
    print(f"{pk:<16}{p['get_by_case_id']:>14,.1f} RU{p['list_by_applicant']:>17,.1f} RU"
          f"{p['filter_by_status']:>16,.1f} RU")

print(f"""
Nothing wins all three. That is not a shortcoming of Cosmos — it is what a
partition key IS. It makes one access pattern a point read and every other
one a scan.

A cross-partition query costs RU per document EXAMINED, not per document
returned. Asking for a handful of pending cases reads all {N_CASES:,}.
""")

## 6 · What that costs per month

Your measured RU, multiplied by the volumes from the brief.

In [ ]:
MONTHLY = {"get_by_case_id": 40_000, "list_by_applicant": 3_000,
           "filter_by_status": 6_000, "writes": 16_000}
PRICE = {"serverless_per_1m_ru": 0.28, "provisioned_per_100ru_hour": 0.008}

def fmt_ru(x):
    return f"{x/1_000_000:.2f}M" if x >= 1e6 else (f"{x/1000:.1f}k" if x >= 1000 else f"{x:.0f}")

print(f"{'partition key':<16}{'monthly RU':>14}{'serverless':>14}{'vs best':>10}")
print("-" * 56)
totals = {}
for pk in CANDIDATES:
    p = PATTERNS[pk]
    ru = sum(p[k] * MONTHLY[k] for k in
             ("get_by_case_id", "list_by_applicant", "filter_by_status"))
    ru += load_stats[pk]["ru_per_write"] * MONTHLY["writes"]
    totals[pk] = ru
best = min(totals.values())
for pk in CANDIDATES:
    ru = totals[pk]
    print(f"{pk:<16}{fmt_ru(ru):>14}{'$'+format(ru/1e6*PRICE['serverless_per_1m_ru'],',.2f'):>14}"
          f"{ru/best:>9.1f}x")

winner = min(totals, key=totals.get)
worst  = max(totals, key=totals.get)
print(f"""
{winner} costs {totals[worst]/totals[winner]:.0f}x less than {worst} for the SAME workload.

One design decision, taken in an afternoon, with a permanent bill attached —
and the partition key is IMMUTABLE, so changing it later means migrating the
container.

Note these figures scale from {N_CASES:,} documents. At the deck's 4,000 the
cross-partition scans cost {4000/N_CASES:.0f}x more, because a scan is charged
per document examined and your corpus is what it examines.
""")

## 7 · Partition skew — where the hot partition comes from

A container-level RU number hides this completely. Ask Cosmos how the documents
are actually distributed.

In [ ]:
def skew(pk, container):
    if LIVE:
        counts = Counter()
        try:
            for row in q(container, f"SELECT c.{pk} AS k, COUNT(1) AS n FROM c GROUP BY c.{pk}"):
                counts[row["k"]] = row["n"]
        except Exception:
            # GROUP BY is not available on every account tier. Fall back to
            # counting client-side — more RU, same answer.
            for row in q(container, f"SELECT c.{pk} AS k FROM c"):
                counts[row["k"]] += 1
    else:
        counts = Counter(c[pk] for c in CASES)
    total = sum(counts.values()) or 1
    largest = max(counts.values())
    return {"logical_partitions": len(counts), "largest_share": largest/total,
            "top": counts.most_common(5)}

print(f"{'partition key':<16}{'logical partitions':>20}{'largest share':>16}   hottest values")
print("-" * 96)
SKEW = {}
for pk in CANDIDATES:
    s = skew(pk, containers[pk] if LIVE else None)
    SKEW[pk] = s
    hot = "  HOT" if s["largest_share"] > 0.35 else ""
    top = ", ".join(f"{k}={n}" for k, n in s["top"][:3])
    print(f"{pk:<16}{s['logical_partitions']:>20,}{s['largest_share']:>15.0%}{hot}   {top[:44]}")

print("""
HIGH CARDINALITY and EVEN DISTRIBUTION. That is the whole test.

case_id has one logical partition per case, evenly spread by construction.
status has five, and most cases end up complete — so three quarters of your
documents land on one of them.
""")

## 8 · The hot partition — the most confusing symptom in Cosmos

A single **physical** partition cannot exceed roughly **1,000 RU/s**, whatever
you provision at the container level.

So a skewed key gives you `429 Too Many Requests` while the dashboard shows
spare capacity. That is a partition key problem wearing a capacity costume, and
teams respond by provisioning more, which changes nothing.

In [ ]:
OPS_PER_SEC, RU_PER_OP, PROVISIONED = 500, 8.0, 6000
PARTITION_CEILING = 1000

demand = OPS_PER_SEC * RU_PER_OP
print(f"Morning peak: forty handlers open the queue view and it auto-refreshes.")
print(f"{OPS_PER_SEC} ops/sec x {RU_PER_OP:.0f} RU = {demand:,.0f} RU/s demand.")
print(f"You provisioned {PROVISIONED:,} RU/s — {PROVISIONED/demand:.1f}x headroom.\n")

print(f"{'partition key':<16}{'largest partition':>19}{'its demand':>14}"
      f"{'ceiling':>10}{'result':>16}")
print("-" * 76)
for pk in CANDIDATES:
    share = SKEW[pk]["largest_share"]
    hot_demand = demand * share
    result = "429 THROTTLED" if hot_demand > PARTITION_CEILING else "fine"
    print(f"{pk:<16}{share:>18.0%}{hot_demand:>11,.0f} RU/s"
          f"{PARTITION_CEILING:>7,} RU/s{result:>16}")

print(f"""
The container dashboard would show {PROVISIONED - demand:,.0f} RU/s spare the
whole time.

This is why an unusually GOOD number deserves the same suspicion as a bad one.
Plenty of headroom at the container level tells you nothing about whether one
partition is on fire.
""")

## 9 · Document shape — a write-cost decision

Cosmos charges write RU by **document size**. A case changes status four times.

With everything in one document, every status change rewrites the AI narrative
and its citations — which have not changed and never will.

Two more containers, real writes, real RU.

In [ ]:
SHAPE = {}
if LIVE:
    for label, fat in (("fat", True), ("hot_split", False)):
        name = f"{PREFIX}_shape_{label}"
        try:
            db.delete_container(name)
        except exceptions.CosmosResourceNotFoundError:
            pass
        c = db.create_container(id=name, partition_key=PartitionKey(path="/case_id"))
        docs = [make_case(i, random.Random(7), fat=fat) for i in range(min(120, N_CASES))]
        ru = 0.0
        for d in docs:
            c.upsert_item(d); ru += charge(c)
        # then four status updates each, as a real case lifecycle would
        upd_ru = 0.0
        for d in docs:
            for st in ("extracting", "generating", "complete"):
                d["status"] = st
                c.upsert_item(d); upd_ru += charge(c)
        SHAPE[label] = {"kb": len(json.dumps(docs[0]))/1024,
                        "ru_per_write": ru/len(docs),
                        "ru_per_update": upd_ru/(len(docs)*3),
                        "container": name}
        print(f"  {name:<22} {SHAPE[label]['kb']:>5.1f} KB   "
              f"{SHAPE[label]['ru_per_write']:>6.1f} RU/write   "
              f"{SHAPE[label]['ru_per_update']:>6.1f} RU/status-update")
else:
    SHAPE = {"fat":       {"kb": 3.0, "ru_per_write": 15.0, "ru_per_update": 15.0},
             "hot_split": {"kb": 0.5, "ru_per_write": 2.5,  "ru_per_update": 2.5}}
    for k, v in SHAPE.items():
        print(f"  {PREFIX}_shape_{k:<16} {v['kb']:>5.1f} KB   "
              f"{v['ru_per_write']:>6.1f} RU/write   {v['ru_per_update']:>6.1f} RU/update  (simulated)")

f, s = SHAPE["fat"], SHAPE["hot_split"]
monthly_fat   = f["ru_per_update"] * MONTHLY["writes"]
monthly_split = s["ru_per_update"] * MONTHLY["writes"]
print(f"""
{'':<26}{'doc size':>10}{'RU/update':>12}{'monthly write RU':>19}
{'-'*68}
{'one fat document':<26}{f['kb']:>8.1f}KB{f['ru_per_update']:>12.1f}{fmt_ru(monthly_fat):>19}
{'hot + cold split':<26}{s['kb']:>8.1f}KB{s['ru_per_update']:>12.1f}{fmt_ru(monthly_split):>19}

{f['ru_per_update']/s['ru_per_update']:.0f}x cheaper per status change.

The cost of splitting is ONE extra read when somebody wants the full result —
and that is a point read, the cheapest thing Cosmos does.

Separate what changes from what does not.
""")

## 10 · The change feed — what your audit trail will not see

**This is the cell to raise in Sprint 0 terms today**, because Sprint 2 is
where it gets discovered otherwise, and by then the audit trail is a commitment
rather than a design option.

The default change feed gives you the **latest version of a changed document,
not every version it passed through.**

Move real cases through their lifecycle, then poll the feed and count.

In [ ]:
LIFECYCLE = ["pending", "extracting", "generating", "complete"]
N_LIFECYCLE = 30

if LIVE:
    name = f"{PREFIX}_changefeed"
    try:
        db.delete_container(name)
    except exceptions.CosmosResourceNotFoundError:
        pass
    cf = db.create_container(id=name, partition_key=PartitionKey(path="/case_id"))

    # Take a continuation token from NOW, so we only see what we are about to do.
    _ = read_changefeed(cf, from_beginning=False)
    time.sleep(1)

    transitions = 0
    for i in range(N_LIFECYCLE):
        cid = f"CLM-2026-{9000+i}"
        for st in LIFECYCLE:
            cf.upsert_item({"id": cid, "case_id": cid, "status": st,
                            "updated_at": datetime.now(timezone.utc).isoformat()})
            transitions += 1
    time.sleep(3)          # let the feed settle

    observed = read_changefeed(cf, from_beginning=True)
    seen_states = Counter(d.get("status") for d in observed)
else:
    transitions = N_LIFECYCLE * len(LIFECYCLE)
    observed = [{"status": "complete"} for _ in range(N_LIFECYCLE)]
    seen_states = Counter({"complete": N_LIFECYCLE})

lost = transitions - len(observed)
print(f"{'':<34}{'transitions':>13}{'observed':>11}{'LOST':>10}")
print("-" * 70)
print(f"{'latest_version  (the default)':<34}{transitions:>13}{len(observed):>11}"
      f"{lost:>7} ({lost/transitions:.0%})")
print(f"\nstates the feed actually reported: {dict(seen_states)}")

print(f"""
Every case passed through {len(LIFECYCLE)} states. The feed reported {len(observed)}
changes, and they are almost all 'complete'.

The intermediate states HAPPENED. They were written, they were real, and they
were never captured — because a case that moves through its lifecycle faster
than the polling interval is observed once, in whatever state it finished in.

FOR A STATUS DASHBOARD    this is fine. Nobody needs the intermediate states
                          of a case that completed thirty seconds ago.

FOR YOUR SPRINT 0 AUDIT   it is a hole. You cannot evidence a state you never
TRAIL                     recorded, and "the system moved through generating"
                          is exactly what an investigation asks about.
""")

### Two ways out, and they cost different things

**All-versions mode** — every version reaches the consumer, no application
change, more feed volume and more cost. Availability varies by account
configuration and API version; check yours rather than assuming.

**An explicit transition record** — an append-only event written on every state
change. Never mutated, so nothing can be lost, and you control exactly what is
captured. One extra small write per transition.

> The second is what an auditor actually wants, because it is a **log** rather
> than a stream of snapshots. A change feed tells you what a document looks like
> **now**; a transition record tells you **what happened and in what order**.
> Only one of those survives an investigation.

It also fits the Sprint 0 Day 6 ordering constraint — the audit record is
written **before** the result is released. A transition record can be. A change
feed observation cannot, because it happens after the write.

The next cell measures what that costs.

In [ ]:
if LIVE:
    name = f"{PREFIX}_transitions"
    try:
        db.delete_container(name)
    except exceptions.CosmosResourceNotFoundError:
        pass
    tr = db.create_container(id=name, partition_key=PartitionKey(path="/case_id"))
    ru = 0.0
    for i in range(N_LIFECYCLE):
        cid = f"CLM-2026-{9000+i}"
        prev = None
        for st in LIFECYCLE:
            tr.create_item({"id": f"{cid}:{st}:{uuid.uuid4().hex[:6]}",
                            "case_id": cid, "from_state": prev, "to_state": st,
                            "at": datetime.now(timezone.utc).isoformat(),
                            "actor": "pipeline"})
            ru += charge(tr); prev = st
    per = ru / (N_LIFECYCLE * len(LIFECYCLE))
else:
    per, ru = 5.2, 5.2 * N_LIFECYCLE * len(LIFECYCLE)

monthly = per * MONTHLY["writes"]
print(f"""transition records written: {N_LIFECYCLE * len(LIFECYCLE)}
RU per transition record:   {per:.1f}
at {MONTHLY['writes']:,} status changes a month: {fmt_ru(monthly)} RU
                                    ~${monthly/1e6*PRICE['serverless_per_1m_ru']:.2f} on serverless

That is what a complete, append-only audit trail costs. Put that number in the
ADR next to the sentence about what the change feed will not capture, and let
somebody decide with both in front of them.
""")

## 11 · Serverless or provisioned

In [ ]:
total_ru = totals[winner]
avg_ru_s = total_ru / (30*24*3600)
prov_ru  = max(400, int(round(avg_ru_s*3/100))*100)      # provision for ~3x average
serverless_cost = total_ru/1e6*PRICE["serverless_per_1m_ru"]
prov_cost = prov_ru/100*PRICE["provisioned_per_100ru_hour"]*24*30

print(f"""with your best partition key ({winner}):

  monthly RU              {fmt_ru(total_ru):>12}
  average RU/s            {avg_ru_s:>12,.1f}
  serverless              ${serverless_cost:>11,.2f}
  provisioned ({prov_ru} RU/s) ${prov_cost:>11,.2f}   <- the floor you pay regardless

Provisioned has a 400 RU/s minimum. At this volume you are nowhere near it,
so you would be paying for capacity you never use.

THE FINDING THAT IS EASY TO MISS: with a GOOD partition key serverless is far
cheaper than the provisioned floor. With a BAD one you consume your way up to
the floor and the two look level — so the bad design does not appear expensive
on a provisioned plan. It quietly eats the headroom you needed for growth.

For the POC: serverless. It also gives you a real RU figure to size a
provisioned tier with later, which beats a guess.
""")

## 12 · Clean up — run this

Deletes every container this notebook created. On a provisioned account each
one may carry a 400 RU/s minimum, so leaving five behind is a real monthly
charge for nothing.

In [ ]:
if LIVE:
    deleted = []
    for c in list(db.list_containers()):
        if c["id"].startswith(f"{PREFIX}_"):
            db.delete_container(c["id"]); deleted.append(c["id"])
    print("deleted:", ", ".join(deleted) if deleted else "nothing found")

    DROP_DATABASE = False      # set True to remove forge_day18 entirely
    if DROP_DATABASE:
        client.delete_database(DB_NAME)
        print(f"database {DB_NAME} deleted")
    else:
        print(f"database {DB_NAME} kept (empty). Set DROP_DATABASE=True to remove it.")
else:
    print("simulated — nothing to clean up")

## 13 · Export and the ADR

In [ ]:
design = {
  "mode": "live" if LIVE else "simulated",
  "n_cases_measured": N_CASES,
  "partition_key": {
      "chosen": winner,
      "rejected": {pk: {"monthly_ru": round(totals[pk]),
                        "vs_chosen": round(totals[pk]/totals[winner], 1)}
                   for pk in CANDIDATES if pk != winner},
      "why": "highest cardinality, evenest distribution, and it makes the "
             "40,000/month pattern a point read"},
  "measured_ru_per_pattern": {pk: {k: round(v, 2) for k, v in PATTERNS[pk].items()}
                              for pk in CANDIDATES},
  "partition_skew": {pk: {"logical_partitions": SKEW[pk]["logical_partitions"],
                          "largest_share": round(SKEW[pk]["largest_share"], 3)}
                     for pk in CANDIDATES},
  "document_shape": {k: {"kb": round(v["kb"], 2),
                         "ru_per_update": round(v["ru_per_update"], 2)}
                     for k, v in SHAPE.items()},
  "change_feed": {"mode": "latest_version",
                  "transitions": transitions, "observed": len(observed),
                  "lost_pct": round(lost/transitions, 3),
                  "mitigation": "append-only transition record",
                  "transition_ru_each": round(per, 2)},
  "throughput": {"poc": "serverless", "monthly_ru": round(total_ru),
                 "provisioned_floor_ru_s": 400},
  "review_triggers": ["a new access pattern", "volume beyond the tested range",
                      "partition skew rising"],
}
from pathlib import Path
Path("cosmos_design.json").write_text(json.dumps(design, indent=2))
print(json.dumps(design, indent=2)[:1800])
print("\n... written to cosmos_design.json")

try:
    from google.colab import files
    files.download("cosmos_design.json")
except Exception:
    pass

## 14 · Questions for the ADR

1. **The partition key is immutable.** Changing it means migrating the
   container. Are you confident enough in the access patterns to commit — and
   if not, is now the cheapest moment you will ever have to be wrong?

2. **Which access pattern did you optimise for, and what is the second one
   costing you?** You measured both. Put both numbers in.

3. **Your hot document and your cold document** — what happens if the write of
   one succeeds and the other fails? Cosmos transactional batches work within a
   single logical partition; are your two documents in the same one?

4. **The change feed observes the latest version, not every version.** Does
   your audit trail depend on the states it will miss, and have you costed the
   transition record?

5. **Serverless or provisioned, and at what volume does that answer change?**
   You now have a measured RU figure rather than a guess.

6. **What is your review trigger** for revisiting this design?

---

### Name the rejected candidates

Row one of your ADR is the one you cannot revisit. In two years somebody will
ask why `case_id`, and *"it seemed right"* is not an answer.

You have the numbers. Write them down.

---

### One honest limitation

You measured **{N} documents** from a Colab VM, against one account, at low
concurrency, on one day.

Cross-partition scans scale with corpus size, so at 4,000 documents the numbers
in the middle column get roughly ten times worse — which strengthens the
conclusion rather than weakening it. But re-measure at your real volume before
any figure goes in a contract, and re-measure from inside your own network
before you quote a latency.